# Passing Networks (EXTRA)

**Chapter 3: Exploratory Data Analysis in Soccer**

## Learning Objectives

By the end of this notebook, you will be able to:
- Calculate average player positions from event data
- Compute pass frequencies between players
- Create passing network visualizations
- Interpret tactical structures from passing networks
- Identify key playmakers and passing combinations

## Prerequisites

- Completed core notebooks 01-04
- Completed extra notebook 05 (mplsoccer)
- Basic understanding of graph/network concepts helpful

## What is a Passing Network?

A passing network is a visualization that shows:
- **Nodes**: Players positioned at their average location on the pitch
- **Edges**: Lines between players representing pass frequency
- **Node size**: Often represents number of passes made
- **Edge thickness**: Represents how often two players passed to each other

This reveals the tactical structure and key relationships within a team.

## Setup and Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from mplsoccer import Pitch

# Set style
sns.set_style("white")
plt.rcParams['figure.figsize'] = (12, 8)

## Load Event Data

In [ ]:
# Load events from a match
BASE = Path("open-data/data")
match_id = 22921  # France vs Korea Republic

events = json.load(open(BASE / f"events/{match_id}.json", "r"))
events_df = pd.json_normalize(events, sep="_")
events_df["match_id"] = match_id

print(f"Loaded {len(events_df)} events")

## Step 1: Extract Passes for One Team

In [ ]:
# Select team
team_name = "France Women's"

# Filter for completed passes by this team
passes = events_df.query(
    "type_name == 'Pass' and team_name == @team_name"
).copy()

# Mark completed passes (NaN outcome = completed)
passes["completed"] = passes["pass_outcome_name"].isna()

# Keep only completed passes for network analysis
passes = passes[passes["completed"]]

print(f"{team_name} completed {len(passes)} passes")

## Step 2: Extract Player and Location Information

In [ ]:
# Extract passer (player who made the pass)
passes["passer"] = passes["player_name"]

# Extract receiver (player who received the pass)
# This is usually in pass_recipient_name
if "pass_recipient_name" in passes.columns:
    passes["receiver"] = passes["pass_recipient_name"]
else:
    print("Warning: pass_recipient_name not found")
    # Alternative: try to infer from next event
    passes["receiver"] = None

# Extract pass start location
if "location" in passes.columns:
    passes[["x", "y"]] = passes["location"].apply(pd.Series)
elif "location_0" in passes.columns:
    passes["x"] = passes["location_0"]
    passes["y"] = passes["location_1"]

# Remove passes with missing data
passes = passes.dropna(subset=["passer", "receiver", "x", "y"])

print(f"After cleaning: {len(passes)} passes with complete information")
passes[["passer", "receiver", "x", "y"]].head()

## Step 3: Calculate Average Player Positions

For each player, we calculate their average x and y coordinates across all their passes.

In [ ]:
# Calculate average position for each player
player_positions = (
    passes.groupby("passer")
    .agg(
        avg_x=("x", "mean"),
        avg_y=("y", "mean"),
        num_passes=("passer", "count")
    )
    .reset_index()
)

print("\nAverage Player Positions:")
print(player_positions.sort_values("num_passes", ascending=False).head(10))

## Step 4: Calculate Pass Frequencies Between Players

Count how many times each pair of players passed to each other.

In [ ]:
# Count passes between each pair of players
pass_between = (
    passes.groupby(["passer", "receiver"])
    .size()
    .reset_index(name="pass_count")
)

# Filter for significant connections (e.g., at least 3 passes)
min_passes = 3
pass_between = pass_between[pass_between["pass_count"] >= min_passes]

print(f"\nFound {len(pass_between)} significant passing connections (>= {min_passes} passes)")
print("\nTop 10 Passing Combinations:")
print(pass_between.sort_values("pass_count", ascending=False).head(10))

## Step 5: Create the Passing Network Visualization

In [ ]:
# Create pitch
pitch = Pitch(pitch_color='#22312b', line_color='#efefef', linewidth=2)
fig, ax = pitch.draw(figsize=(14, 10))

# Plot passing lines (edges)
for _, row in pass_between.iterrows():
    passer = row["passer"]
    receiver = row["receiver"]
    count = row["pass_count"]
    
    # Get positions
    passer_pos = player_positions[player_positions["passer"] == passer]
    receiver_pos = player_positions[player_positions["passer"] == receiver]
    
    if len(passer_pos) > 0 and len(receiver_pos) > 0:
        x1, y1 = passer_pos.iloc[0][["avg_x", "avg_y"]]
        x2, y2 = receiver_pos.iloc[0][["avg_x", "avg_y"]]
        
        # Line thickness based on pass count
        linewidth = count / 2  # Scale as needed
        
        pitch.lines(x1, y1, x2, y2, ax=ax, 
                   color='white', alpha=0.3, lw=linewidth, zorder=1)

# Plot player nodes
# Size nodes by number of passes
node_sizes = player_positions["num_passes"] * 3  # Scale for visibility

pitch.scatter(player_positions["avg_x"], player_positions["avg_y"],
             s=node_sizes, color='#00ff85', edgecolors='white', linewidth=2,
             ax=ax, zorder=3)

# Add player names (for top passers only, to avoid clutter)
top_passers = player_positions.nlargest(11, "num_passes")  # Top 11 (starting XI)
for _, player in top_passers.iterrows():
    # Extract last name for cleaner labels
    name = player["passer"].split()[-1] if " " in player["passer"] else player["passer"]
    ax.text(player["avg_x"], player["avg_y"] - 3, name,
           ha='center', va='top', fontsize=9, color='white', weight='bold')

plt.title(f"Passing Network — {team_name}", fontsize=16, pad=20, color='white')
plt.tight_layout()
plt.show()

**How to Interpret This Visualization:**

1. **Node Position**: Shows where each player operated on average
   - Reveals the team's formation and shape
   - Defensive players appear lower, attackers higher

2. **Node Size**: Indicates passing volume
   - Larger nodes = more passes made
   - Identifies key distributors

3. **Edge Thickness**: Shows passing frequency between players
   - Thicker lines = more frequent connections
   - Reveals preferred passing combinations

4. **Network Structure**: Shows tactical organization
   - Central players with many connections are key playmakers
   - Isolated players may indicate tactical issues
   - Dense connections suggest good team cohesion

## Advanced Analysis: Identifying Key Playmakers

We can quantify player importance using network metrics.

In [ ]:
# Calculate degree centrality (number of unique passing partners)
passer_connections = pass_between.groupby("passer")["receiver"].nunique().reset_index()
passer_connections.columns = ["player", "num_connections"]

receiver_connections = pass_between.groupby("receiver")["passer"].nunique().reset_index()
receiver_connections.columns = ["player", "num_connections_received"]

# Merge with pass counts
player_metrics = player_positions.merge(passer_connections, left_on="passer", right_on="player", how="left")
player_metrics = player_metrics.merge(receiver_connections, left_on="passer", right_on="player", how="left")

# Fill NaN with 0
player_metrics = player_metrics.fillna(0)

# Calculate total connections
player_metrics["total_connections"] = (
    player_metrics["num_connections"] + player_metrics["num_connections_received"]
)

print("\nKey Playmakers (by connections and passes):")
print(player_metrics[["passer", "num_passes", "total_connections"]]
      .sort_values("total_connections", ascending=False)
      .head(10))

## Summary

In this notebook, we learned how to:

1. **Extract passing data** from event logs
2. **Calculate average player positions** from pass locations
3. **Compute pass frequencies** between players
4. **Create passing network visualizations** using mplsoccer
5. **Interpret tactical structures** from network patterns
6. **Identify key playmakers** using network metrics

Passing networks are powerful tools for:
- Understanding team shape and formation
- Identifying key distributors and playmakers
- Revealing tactical patterns and preferred combinations
- Comparing different teams' playing styles
- Analyzing how tactics change across matches

## Next Steps

- Apply this to the Japan case study (extra notebook 07)
- Compare passing networks between different teams
- Analyze how networks change in different match situations (winning vs losing)
- Explore defensive action networks (tackles, interceptions)

## Practice Exercises

1. **Compare Two Teams**: Create side-by-side passing networks for both teams in the same match
2. **Time-Based Networks**: Create separate networks for first half vs second half
3. **Weighted Centrality**: Calculate betweenness centrality to find players who bridge different parts of the team
4. **Progressive Passes**: Filter for only forward/progressive passes and create a network
5. **Defensive Network**: Create a network of defensive actions (tackles, interceptions, clearances)